In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from alibi.explainers import AnchorTabular
from sklearn.metrics import accuracy_score
import pandas as pd

C:\Users\chalo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### prepare the dataset

In [2]:
data = load_iris()
X, Y = data.data, data.target
feature_names = data.feature_names
class_names = data.target_names

# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

### put the data in a dataframe for easier visualization

In [3]:
columns=['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
df = pd.DataFrame(x_test, columns=columns)
df['target'] = y_test
df.head(2)

,sepal_length,sepal_width,petal_length,petal_width,target
0,6.1,2.8,4.7,1.2,1
1,5.7,3.8,1.7,0.3,0


### train a random forest classifier

In [4]:
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(x_train, y_train)
predictions =model.predict(x_test)
accuracy=accuracy_score(y_test, predictions)
print("Accuracy: {:.2f}%".format(accuracy*100))

Accuracy: 100.00%


### print the targets and prediction

In [5]:
targets=list(data.target_names)
print(targets)

pred=model.predict(x_test[0].reshape(1,-1))
print("Actual {} ; Predicted {}".format(targets[y_test[0]],
targets[pred[0]]))

['setosa', 'versicolor', 'virginica']
Actual versicolor ; Predicted versicolor


### initialize the Anchor explainer and fit the explainer to the data

In [6]:
explainer = AnchorTabular(model.predict, feature_names=feature_names, categorical_names={})

explainer.fit(x_test, disc_perc=(20,40, 60, 60)) # Discretize continuous variables for explanation

AnchorTabular(meta={
  'name': 'AnchorTabular',
  'type': ['blackbox'],
  'explanations': ['local'],
  'params': {'seed': None, 'disc_perc': (20, 40, 60, 60)},
  'version': '0.9.6'}
)

### pick a sample from the test set and explain the prediction

In [7]:
sample = x_test[1].reshape(1,-1)
print(f"Model prediction: {class_names[model.predict(sample)[0]]}")

#Explain the model’s prediction for the selected instance
explanation = explainer.explain(sample, threshold=0.95)

#Print the anchor explanation
print(f'Anchor: {" AND ".join(explanation.anchor)}')
print(f'Precision: {explanation.precision}')
print(f'Coverage: {explanation.coverage*100}%')

Model prediction: setosa
Anchor: petal width (cm) <= 0.30
Precision: 1.0
Coverage: 26.96%


### view the sample

In [8]:
df.iloc[1].head()

sepal_length    5.7
sepal_width     3.8
petal_length    1.7
petal_width     0.3
target          0.0
Name: 1, dtype: float64